# Preserve CSA full XE10 — exact Output139
**Accelerator NONE, Internet ON, no Add Input.** Enable Secret `KAGGLE_API_TOKEN`; if named `codex`, edit SECRET_NAME.
Downloads best.bin (validated epoch9) and checkpoint.bin (trained epoch10), receipts and original review ZIP from exact139.
Checks every original byte SHA and copies unchanged into a NEW Private Dataset. Never load weights/run models.
Keep source139 until Dataset finishes processing. The review ZIP contains all other logs/captions/references.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/139'
OWNER='ducanh2403'
SLUG='csa-full-xe10-v139'
TITLE='CSA full XE10 v139 checkpoints dataset'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/csa_full_xe10_v139_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/139':raise ValueError('This notebook pins139; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/139', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v139', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v139; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/139', 'files': {'result.json': '723e3897a7b62dfeddbcaeb5711e47409feec8b3042d0b99f454f8e215d0295e', 'progress.json': 'c8757e5f6f20c857c2debbeedf5042a65a661dd207ea9783bd9fdd548418b097', 'input_identity.json': 'b7d1f08ecc045ea6779c5c04776b52d5b92933a88d58fe792b3f897c5cd98c5b', 'best_receipt.json': '9bc44a09f17074b5087cbf13e7b13461048e70e0aaee3f89a12f6834517599f9', 'checkpoint_receipt.json': 'a46cc05fe4af3fe52699dc0a557a816d33cbb08fa69694eefcb1e84cfcc52f68', 'best.bin': 'f09e11c374477be93cc142a238dbd0e206060f33fa4fab5988e767ea7d259fe6', 'checkpoint.bin': '272f843a21b3a63c61323f748ed916fa3ae0cf2777549061439c07b410006950', 'csa_full_xe_10_v2_review.zip': '7ff1ba033f560b2a0565cb30d7bae137eba8fca9a59db45e9e947b7cb9db286f'}}
SOURCE_CODE='"""Exact139 checkpoint transport; byte checks only, never deserialize weights."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024**2),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef package(download,destination,inventory):\n    download,destination=Path(download),Path(destination)\n    if inventory[\'kernel_version\']!=\'ducanh2403/2-vi-cai-kia-qua-nhieu/139\':raise ValueError(\'Expected exact version139\')\n    if destination.exists():raise ValueError(\'Preserve existing destination; choose a fresh path\')\n    paths={}\n    for name,h in inventory[\'files\'].items():\n        if Path(name).name!=name:raise ValueError(\'Unexpected payload path\')\n        matches=[p for p in download.rglob(name) if p.is_file() and sha(p)==h]\n        if len(matches)!=1:raise ValueError(f\'Output139: expected one verified {name}, got {len(matches)}; no latest fallback\')\n        paths[name]=matches[0]\n    read=lambda n:json.loads(paths[n].read_text(encoding=\'utf-8\'))\n    r=read(\'result.json\');best=read(\'best_receipt.json\');last=read(\'checkpoint_receipt.json\')\n    if r[\'status\']!=\'FULL_XE_10_COMPLETE\' or (r[\'completed_epochs\'],r[\'updates\'],r[\'best_epoch\'])!=(10,177110,9):raise ValueError(\'Wrong training endpoint\')\n    if (best[\'validated_epoch\'],best[\'updates\'],best[\'sha256\'])!=(9,159399,inventory[\'files\'][\'best.bin\']):raise ValueError(\'Wrong best checkpoint\')\n    if (last[\'epoch\'],last[\'updates\'],last[\'sha256\'])!=(11,177110,inventory[\'files\'][\'checkpoint.bin\']):raise ValueError(\'Wrong latest checkpoint\')\n    destination.mkdir(parents=True)\n    for name,p in paths.items():\n        shutil.copy2(p,destination/name)\n        if sha(destination/name)!=inventory[\'files\'][name]:raise ValueError(\'Copy bytes differ:\'+name)\n    receipt=dict(status=\'CSA_XE139_TRANSPORT_COMPLETE\',source=inventory[\'kernel_version\'],files_sha256=inventory[\'files\'],\n        best_validated_epoch=9,latest_trained_epoch=10,weights_deserialized=False,\n        also_requires=[\'ducanh2403/csa-full-roi-karpathy-v134\',\'ducanh2403/csa-semantic-banks-v130\'])\n    (destination/\'transport_receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return receipt\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='c61d71038e05b9803963a0a0e1f162c8058de0ddfac0222b61098d48d4b5100f'
exec(compile(SOURCE_CODE,'package_xe139.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<8*1024**3:raise RuntimeError('Need8GiB free for download and unchanged copy.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa-xe139-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Downloading EXACT:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
report=package(DOWNLOAD,DESTINATION,INVENTORY)
print('Both checkpoints and original review verified/copied unchanged.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private checkpoints from exact notebook139: CSA XE10 latest and validation-best epoch9, original binary bytes, receipts and complete small review ZIP. Retains upstream model/data terms. No model/training rerun.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; choose a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('Creation accepted; Dataset may still be processing. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status unconfirmed; check Dataset page before retrying.',flush=True)


Send the actual mount after creation finishes. Expected:
`/kaggle/input/datasets/ducanh2403/csa-full-xe10-v139`.
Keep ROI134 and semantic banks130 separately for any later continuation. This notebook preserves artifacts only; it does not extend training.
